## imports

In [1]:
#
# !sudo apt-get update > /dev/null 2>&1
# !sudo apt-get install -y xvfb python-opengl ffmpeg > /dev/null 2>&1
# !pip install rarfile --quiet
# !pip install stable-baselines3[extra] ale-py==0.7.4 --quiet
# !pip install box2d-py --quiet
# !pip install gym pyvirtualdisplay --quiet
# !pip install pyglet --quiet
# !pip install piglet --quiet

In [2]:
#from pyvirtualdisplay import Display
#display = Display(visible=0, size=(1400, 900))
#display.start()

In [3]:
import gym
#from gym.wrappers import Monitor
import torch
import matplotlib.pyplot as plt
import numpy as np

# For visualization
from gym.wrappers.monitoring import video_recorder
from IPython.display import HTML
from IPython import display 
import glob

## LunarLander

### 환경만들기 

`-` 환경을 만드는 방법은 아래와 같다. 

In [4]:
env = gym.make('LunarLander-v2')

`-` 환경에 대한 기본 정보를 조사하여 보자. 

In [5]:
env.observation_space

Box([-inf -inf -inf -inf -inf -inf -inf -inf], [inf inf inf inf inf inf inf inf], (8,), float32)

In [6]:
env.action_space

Discrete(4)

- 이 환경에서 유효한 action은 4개이다. (=에이전트는 4개의 action을 할 수 있다.)

### 환경관찰 

`-` 환경관찰 

In [7]:
env.reset()

array([ 0.00171671,  1.416026  ,  0.17387414,  0.22693141, -0.0019825 ,
       -0.03938509,  0.        ,  0.        ], dtype=float32)

In [8]:
env.step(0) # state, reward, done, _ 

(array([ 0.00343342,  1.4205539 ,  0.17363918,  0.20123672, -0.00392862,
        -0.03892545,  0.        ,  0.        ], dtype=float32),
 1.3612878848733203,
 False,
 {})

`-` action

In [9]:
env.action_space.sample()

3

- int형으로 전달 

`-` action -> nextstate, reward, done

In [10]:
env.step(env.action_space.sample())

(array([ 0.00505457,  1.4244807 ,  0.16166165,  0.17452918, -0.00347052,
         0.00916248,  0.        ,  0.        ], dtype=float32),
 2.4124276547246395,
 False,
 {})

### Play Game

`-` 랜덤액션을 연속적으로 생성하고 그 결과를 기록해보자. 

In [11]:
states = []
actions = []
rewards = []
next_states = []
dones = []

In [12]:
_state1 = env.reset()
for t in range(1500):
    _action = env.action_space.sample() 
    _state2, _reward, _done, _ = env.step(_action) 
    ## save code 
    states.append(_state1.tolist()) 
    actions.append(_action)
    rewards.append(_reward)
    next_states.append(_state2.tolist())
    dones.append(_done)
    ## save code end 
    _state1 = _state2 
    if _done:
        break

`-` 모인 히스토리를 확인해보자. 

In [13]:
len(states), len(actions), len(next_states), len(rewards), len(dones)

(92, 92, 92, 92, 92)

### Qnetwork 설계

`-` 네트워크의 목적: 내가 여기서 뭘 해야하는지 알려줘! = 내가 이 상태에서, 어떠한 액션을 해야하는지 알려줘 $\to$ 8개의 상태를 입력으로 받으면 4개의 액션에 대한 좋은 정도를 숫자로 표현하는 어떠한 함수를 만들자. 

`-` net 설계 

In [14]:
net = torch.nn.Sequential(
    torch.nn.Linear(in_features=8, out_features=128),
    torch.nn.ReLU(),
    torch.nn.Linear(in_features=128, out_features=64),
    torch.nn.ReLU(),
    torch.nn.Linear(in_features=64, out_features=32),
    torch.nn.ReLU(),
    torch.nn.Linear(in_features=32, out_features=4)
)
net

Sequential(
  (0): Linear(in_features=8, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=4, bias=True)
)

In [15]:
net(torch.tensor(states))

tensor([[0.0962, 0.0421, 0.0267, 0.0832],
        [0.0961, 0.0426, 0.0272, 0.0827],
        [0.0963, 0.0429, 0.0272, 0.0829],
        [0.0961, 0.0431, 0.0274, 0.0825],
        [0.0958, 0.0427, 0.0273, 0.0819],
        [0.0955, 0.0427, 0.0276, 0.0814],
        [0.0960, 0.0427, 0.0274, 0.0825],
        [0.0957, 0.0424, 0.0274, 0.0820],
        [0.0955, 0.0423, 0.0275, 0.0815],
        [0.0956, 0.0421, 0.0275, 0.0817],
        [0.0957, 0.0419, 0.0274, 0.0818],
        [0.0954, 0.0419, 0.0277, 0.0813],
        [0.0951, 0.0416, 0.0276, 0.0805],
        [0.0955, 0.0415, 0.0276, 0.0813],
        [0.0951, 0.0411, 0.0276, 0.0805],
        [0.0944, 0.0407, 0.0277, 0.0798],
        [0.0952, 0.0407, 0.0277, 0.0807],
        [0.0956, 0.0410, 0.0276, 0.0810],
        [0.0948, 0.0406, 0.0278, 0.0801],
        [0.0952, 0.0410, 0.0278, 0.0803],
        [0.0957, 0.0407, 0.0275, 0.0812],
        [0.0948, 0.0403, 0.0278, 0.0801],
        [0.0953, 0.0401, 0.0275, 0.0810],
        [0.0957, 0.0397, 0.0271, 0

### Policy 설계

`-` 네트워크의 의미

In [16]:
states[0],states[1]

([0.0018875121604651213,
  1.4094456434249878,
  0.1911763697862625,
  -0.06553708016872406,
  -0.002180446870625019,
  -0.043304335325956345,
  0.0,
  0.0],
 [0.0038929940201342106,
  1.4089620113372803,
  0.202154278755188,
  -0.02149556204676628,
  -0.00376334972679615,
  -0.03166034072637558,
  0.0,
  0.0])

In [17]:
net(torch.tensor(states[0:2]))

tensor([[0.0962, 0.0421, 0.0267, 0.0832],
        [0.0961, 0.0426, 0.0272, 0.0827]], grad_fn=<AddmmBackward0>)

- 상태0에서는 액션0이, 상태1에서도 액션0이 가장 좋다는 의미 (왜? q-value가 젤 높으니까..)

`-` 따라서 Agent는 아래와 같이 행동해야 한다. (네트워크가 잘 학습되었다는 전제가 필요함) 
- state[0] -> action = 0 
- state[1] -> action = 0 

In [18]:
net(torch.tensor(states[0:2])).max(axis=1)

torch.return_types.max(
values=tensor([0.0962, 0.0961], grad_fn=<MaxBackward0>),
indices=tensor([0, 0]))

In [19]:
net(torch.tensor(states[0:2])).max(axis=1)[1]

tensor([0, 0])

`-` 네트워크가 있으므로 이제 어떠한 state에 대해서도 뭘 해야할지 (=어떤 액션을 해야할지) 알 수 있다. 

In [20]:
_state1 # 어떤 state에 대해서도.. 

array([ 1.5678950e-01, -4.2777721e-02,  1.0349611e-01,  1.7614622e-05,
        2.5364620e-04,  6.8085797e-07,  1.0000000e+00,  1.0000000e+00],
      dtype=float32)

In [21]:
net(torch.tensor(_state1)) # q-value를 계산할 수 있고 

tensor([0.1092, 0.0469, 0.0428, 0.0855], grad_fn=<AddBackward0>)

In [22]:
int(torch.argmax(net(torch.tensor(_state1)))) # 그래서 다음에 우리가 어떤행동을 해야할 지 알 수 있음

0

### 학습

`-` 네트워크를 학습시키자. 

In [23]:
#net.to("cuda:0")

In [24]:
scores=[]
playtimes=[] 
eps = 1
opt = torch.optim.Adam(net.parameters(),lr=1/10000)

In [25]:
for epsd in range(1,2001): # 게임 2000회 시켜줌.. 
    state1 = env.reset() # 환경리셋 + 초기화된 환경을 state라는 변수에 저장 
    score = 0 
    for t in range(500): # 게임1회당 max 1000프레임만 할 수 있음
        # (step1) Agent: action 
        if np.random.rand() < eps: 
            action = env.action_space.sample() # 랜덤액션을 뽑음 
        else:
            action = int(torch.argmax(net(torch.tensor(state1)))) # 네트워크가 알려주는 action을 뽑음 

        # (step2) Agent -> Env // Env -> Agent 
        state2, reward, done, _ = env.step(action) # 액션을 환경에 전달 -> (next_state, reward, done) 을 받음 
        
        # (step3) Agnet: 데이터 저장 
        states.append(state1.tolist())
        actions.append(action)
        rewards.append(reward)
        next_states.append(state2.tolist())
        dones.append(done)
    
        ## 최근 300개의 자료만 준비함. 
        if len(states)>10000:
            _states = torch.tensor(states[-300:])
            _actions = torch.tensor(actions[-300:]).reshape(-1,1)
            _next_states = torch.tensor(next_states[-300:])
            _rewards = torch.tensor(rewards[-300:]).reshape(-1,1)
            _dones = torch.tensor(dones[-300:]).to(torch.float).reshape(-1,1) 
        else:
            _states = torch.tensor(states)
            _actions = torch.tensor(actions).reshape(-1,1)
            _next_states = torch.tensor(next_states)
            _rewards = torch.tensor(rewards).reshape(-1,1)
            _dones = torch.tensor(dones).to(torch.float).reshape(-1,1)

        ## 최근 5000개의 자료에서 128개를 임의로 추출함. 
        _n = len(_states)
        _index = np.random.choice(_n,128) # 128 is batch_size 
        _states = _states[_index]
        _actions = _actions[_index]
        _next_states = _next_states[_index]
        _rewards = _rewards[_index]
        _dones = _dones[_index]
        
        # ## GPU로 이동 
        # _states = _states.to("cuda:0")
        # _actions = _actions.to("cuda:0")
        # _next_states = _next_states.to("cuda:0")
        # _rewards = _rewards.to("cuda:0")
        # _dones = _dones.to("cuda:0")
        
        ## leanrn with pytorch 
        yhat = net(_states).gather(1,_actions) ## (s,a) -> q(s,a) // 내가 현재상태 state에서, 현재 action을 하여 얻을 것이라 예상하는 보상 (net가 알려주는) 
        y = _rewards + 0.99 * net(_next_states).detach().max(1)[0].reshape(-1,1)*(1-_dones) ## 그런데 실제로는 이게 맞다고 봐야지~ 
        loss = torch.mean((y-yhat)**2)
        loss.backward()
        opt.step()
        opt.zero_grad()

        # (step4) Agent: prepare next steps 
        state1 = state2  
        eps = max(0.1, 0.999*eps) 
        score += reward
        
        # terminate 
        if done:
            scores.append(score)
            playtimes.append(t)
            break
            
    print('\rEpisode {}\tScore: {:.2f}\tPlaytime: {:.2f}'.format(epsd, scores[-1],playtimes[-1]), end="")
    if epsd % 100 == 0:
        print('\rEpisode {}\tScore: {:.2f}\tPlaytime: {:.2f}'.format(epsd, np.mean(scores[-100:]),np.mean(playtimes[-100:])))
    if np.mean(scores[-100:])>=200.0:
        print('\nEnvironment solved in {:d} episodes!\tAverage Score: {:.2f}'.format(i_episode, np.mean(scores_window)))
        torch.save(agent.qnetwork_local.state_dict(), 'checkpoint.pth')        
        break

Episode 100	Score: -216.90	Playtime: 174.36
Episode 200	Score: -167.74	Playtime: 192.63
Episode 300	Score: -121.12	Playtime: 210.34
Episode 400	Score: -119.52	Playtime: 213.77
Episode 500	Score: -96.01	Playtime: 228.130
Episode 600	Score: -63.99	Playtime: 211.310
Episode 700	Score: -60.02	Playtime: 186.250
Episode 800	Score: -70.99	Playtime: 201.230
Episode 900	Score: -75.20	Playtime: 195.140
Episode 1000	Score: -23.54	Playtime: 240.02
Episode 1100	Score: 22.72	Playtime: 245.5500
Episode 1200	Score: 73.09	Playtime: 289.2500
Episode 1300	Score: 52.86	Playtime: 266.7400
Episode 1400	Score: 39.07	Playtime: 255.950
Episode 1500	Score: 21.61	Playtime: 234.6000
Episode 1600	Score: 4.65	Playtime: 222.88000
Episode 1700	Score: 2.11	Playtime: 209.34000
Episode 1800	Score: -5.19	Playtime: 235.1900
Episode 1900	Score: 68.83	Playtime: 280.9400
Episode 2000	Score: 51.71	Playtime: 267.4600
